# Day 3 — Forecasting Models: From Baselines to Trees, MLPs, and Sequence Models

**Student Learning Outcomes:**
> **SLO 4:** Implement and evaluate multiple forecasting models (naive baseline, AR,
> tree-based model, MLP, RNN, and LSTM) using time-aware train/test splits.
>
> **SLO 5:** Quantitatively compare forecasting performance using appropriate error metrics
> (e.g., RMSE, MAE) and interpret differences across models.
>
> **SLO 6:** Explain the role of hidden state, recurrence, and gating mechanisms in RNNs
> and LSTMs, and describe how they extend linear memory models.

## Before We Begin

> **Prompt:** Think of a forecast you've made in your own life — a weather guess, a travel
> time estimate, a grade prediction. What information did you use? Did you rely on patterns
> from the past, or did you just go with intuition?
>
> Write 3–5 sentences describing your forecast and whether it was accurate. Then sketch a
> simple timeline showing the information you used versus what you were predicting.
> You will have 7 minutes.

---

### Quick Recap from Days 1 & 2

- **Day 1:** We learned to diagnose time series — check stationarity, transform if needed,
  and read ACF/PACF plots.
- **Day 2:** We built AR models and discovered that any time series can be re-framed as a
  supervised learning problem using a **lag-embedded feature matrix**.

Today we take that lag matrix and plug it into **six different model families**, evaluate
them fairly using error metrics, and develop intuition for *when* each model excels.

In this notebook you will:
1. Create a time-aware (chronological) train/test split
2. Implement and evaluate a naive baseline and AR model
3. Train tree-based models (Random Forest, Gradient Boosting) on the lag matrix
4. Compute and interpret RMSE and MAE
5. Build and train an MLP forecaster in PyTorch
6. Understand the recurrence equation of an RNN and the gating mechanisms of an LSTM
7. Build and compare all six models on the same test set

---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.

In [1]:
"""
Day 3 — Forecasting Models: From Baselines to Trees, MLPs, and Sequence Models
==============================================================================
Student Notebook

SLOs Covered:
  SLO 4 — Implement and evaluate multiple forecasting models (naive baseline, AR,
           tree-based model, MLP, RNN, and LSTM) using time-aware train/test splits.
  SLO 5 — Quantitatively compare forecasting performance using RMSE and MAE.
  SLO 6 — Explain hidden state, recurrence, and gating mechanisms in RNNs and LSTMs.

Dataset : Air Passengers (monthly, 1949–1960)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_pacf

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')

ModuleNotFoundError: No module named 'torch'

---
## Part 1 — Time-Aware Train/Test Splits

### Why can't we shuffle?

In standard supervised learning it is common to shuffle data before splitting into train and
test sets. **For time series, this is forbidden.** Shuffling breaks the temporal ordering
and causes **data leakage**: the model can be trained on observations from the *future*
relative to some test points, giving an artificially optimistic performance estimate.

The correct approach: split at a fixed point in time, keeping all training observations
*before* all test observations.

### 1.1 Load data (same pipeline as Days 1 & 2)

This cell is complete — run it.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df  = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff']       = df['Log_Passengers'].diff()

series = df['Log_Diff'].dropna().values
dates  = df['Date'].iloc[1:].values

print(f'Series length (after diff + dropna): {len(series)}')

In [ ]:
# Lag matrix helper — carried over from Day 2
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series : array-like, shape (T,)
    n_lags : int, number of lag features

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)
    y : np.ndarray, shape (T - n_lags,)
    """
    series = np.array(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags : t])
        y.append(series[t])
    return np.array(X), np.array(y)

### 1.2 Create the chronological split

**Your turn!** Build the lag matrix with `n_lags=12`, then split it into training (first 80%)
and test (last 20%) sets — **in order, without shuffling**.

> 💡 **Syntax reminder:** If `X` is a NumPy array, `X[:split]` and `X[split:]` give the
> first `split` rows and the remaining rows respectively.

In [ ]:
N_LAGS     = 12
TRAIN_FRAC = 0.80

X, y = make_lag_matrix(series, N_LAGS)

# FILL IN: compute the index on where to split (should be an integer, 80% of len(X))
split = ???

# FILL IN: slice X and y into training and test sets chronologically
X_train, X_test = X[???], X[???]
y_train, y_test = y[???], y[???]

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

# Visualise the split
fig, ax = plt.subplots()
train_idx = np.arange(split)
test_idx  = np.arange(split, len(y))
ax.plot(train_idx, y_train, label='Train', color='steelblue')
ax.plot(test_idx,  y_test,  label='Test',  color='crimson')
ax.axvline(split, color='black', linestyle='--', linewidth=1.2, label='Split')
ax.set_title('Time-Aware Train / Test Split — Log-Differenced Passengers')
ax.legend()
plt.tight_layout()
plt.show()

### 💻 Coding Practice

1. Create a test/train function that takes in `TRAIN_FRAC`, `X`, and `y` as input and outputs the training data. Be sure to add appropriate docstring.

In [ ]:
# ADD Question for them to plot the models

---
## Part 2 — Naive Baseline & AR Model

### Why start with a baseline?

Every model evaluation begins with the **simplest possible forecast** — the baseline.
If a complex model can't beat it, the complexity is unjustified.

The **naive last-value baseline** (also called the *persistence forecast*) predicts that
the next value equals the last observed value:

$$\hat{x}_{t+1} = x_t$$

### 2.1 Implement the naive baseline

For each position in the test set, the naive prediction is the immediately preceding value.

In [ ]:
# FILL IN: naive baseline
# The first test prediction = last training value
# Each subsequent prediction = the previous actual test value
naive_preds = np.concatenate([[y_train[-1]], y_test[???]])

print('Naive baseline predictions (first 5):', naive_preds[:5])
print('Actual test values        (first 5):', y_test[:5])

### 2.2 Fit the AR model on training data

> 💡 **Reminder:** We must fit the AR model on **training data only** and use it to
> predict the test window. `AutoReg(series, lags=p).fit()` fits the model;
> `.predict(start, end)` generates forecasts at the original series indices.

In [ ]:
# Subset the series to training observations (include the burn-in lags)
train_series = series[:split + N_LAGS]

# FILL IN: 
ar_model  = ???
ar_result = ar_model.fit()

# Predict the test window (these indices refer to the original series)
ar_start = len(train_series)
ar_end   = ar_start + len(y_test) - 1
ar_preds = ar_result.predict(start=ar_start, end=ar_end, dynamic=False)

print('AR predictions (first 5):   ', ar_preds[:5])
print('Actual test values (first 5):', y_test[:5])

---
## Part 3 — Tree-Based Models

### Introduction to tree-based models

Tree-based models are powerful, non-parametric algorithms that make **no stationarity
assumptions** and can capture non-linear patterns in lag features.

**Random Forest** is an ensemble of $B$ decision trees, each trained on a bootstrap sample
of the training data and a random feature subset. Predictions are averaged:

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(\mathbf{x})$$

**Gradient Boosting** builds trees *sequentially*, where each tree corrects the errors of
the current ensemble. The ensemble is updated as:

$$F_{m}(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot T_m(\mathbf{x})$$

where $\eta$ is the learning rate and $T_m$ fits the residuals of $F_{m-1}$.

> ⚠️ **Important:** Tree-based models are scale-invariant. Do **NOT** scale the lag matrix
> for these models. Scaling is only needed for neural networks (Parts 5–6).

### 3.1 Random Forest

**Your turn!** Fit a `RandomForestRegressor` on the training lag matrix.

> 💡 **Syntax:** `RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)`
> creates the model. `.fit(X, y)` trains it. `.predict(X)` generates predictions.

### Dividir y Confluir 🌊

Work in groups of 2–3. 

| Group | Model | Configuration |
|---|---|---|
| A | Random Forest | `n_estimators=100, max_depth=5` |
| B | Random Forest | `n_estimators=200, max_depth=None` |
| C | Gradient Boosting | `n_estimators=100, learning_rate=0.1` |
| D | Gradient Boosting | `n_estimators=200, learning_rate=0.05` |

Fit your assigned model, compute RMSE and MAE (Part 4), and be ready to share your
results with the class.

In [ ]:
# FILL IN: create and fit a RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=???, max_depth=???, random_state=42)
rf_model.fit(???, ???)
rf_preds = rf_model.predict(???)

print('Random Forest predictions (first 5):', rf_preds[:5])

### 3.2 Gradient Boosting

In [ ]:
# FILL IN: create and fit a GradientBoostingRegressor
gb_model = GradientBoostingRegressor(n_estimators=???, learning_rate=???, random_state=42)
gb_model.fit(???, ???)
gb_preds = gb_model.predict(???)

print('Gradient Boosting predictions (first 5):', gb_preds[:5])

---
## Part 4 — Error Metrics & First Comparison

### Two standard metrics

Let $y_t$ be the actual value and $\hat{y}_t$ the model prediction at test step $t$.
For a test set of size $n$:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}$$

$$\text{MAE}  = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|$$

**RMSE** penalizes large errors more heavily (via squaring). It is preferred when big
mistakes are disproportionately costly.

**MAE** treats all errors equally and is more robust to outliers. It is preferred when
a consistent, interpretable average error is desired.

### 4.1 Implement the metrics function

**Your turn!** Complete the `compute_metrics` function below.

In [ ]:
def compute_metrics(y_true, y_pred, label='Model'):
    """
    Compute and print RMSE and MAE for a set of predictions.

    Parameters
    ----------
    y_true : array-like, actual values
    y_pred : array-like, predicted values
    label  : str, model name for display

    Returns
    -------
    dict with keys 'RMSE' and 'MAE'
    """
    # FILL IN: compute RMSE (hint: np.sqrt + mean_squared_error)
    rmse = ???
    # FILL IN: compute MAE
    mae  = ???
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}


# Evaluate the first four models
results = {}
results['Naive Baseline']    = compute_metrics(y_test, naive_preds,      'Naive Baseline')
results['AR(12)']            = compute_metrics(y_test, ar_preds,  'AR(12)')
results['Random Forest']     = compute_metrics(y_test, rf_preds,         'Random Forest')
results['Gradient Boosting'] = compute_metrics(y_test, gb_preds,         'Gradient Boosting')

In [ ]:
# Visualise predictions on the test set
fig, ax = plt.subplots(figsize=(14, 5))
test_range = np.arange(len(y_test))

ax.plot(test_range, y_test,          label='Actual',           color='black',     linewidth=2)
ax.plot(test_range, naive_preds,     label='Naive Baseline',   color='gray',      linestyle='--')
ax.plot(test_range, ar_preds,        label='AR(12)',           color='steelblue', linestyle='-.')
ax.plot(test_range, rf_preds,        label='Random Forest',    color='forestgreen')
ax.plot(test_range, gb_preds,        label='Gradient Boosting',color='darkorange')

ax.set_title('Test Set Predictions — Log-Differenced Passengers')
ax.set_xlabel('Test Step')
ax.set_ylabel('Log-Differenced Value')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Written Response 4

Based on your RMSE and MAE values, answer in 3–5 sentences:

1. Which model family performs best so far? Does this surprise you? Why or why not?
2. Do the RMSE and MAE rankings agree? If one metric suggested a different winner than
   the other, what would that imply about the error distribution?
3. Why is it essential to compare against the naive baseline rather than reporting
   model metrics in isolation?

**After completing your analysis, share your results with the class** (dividir y confluir
debrief). Were the rankings consistent across groups, or did different hyperparameters
produce different rankings?

> **YOUR ANSWER:**

---
## Part 5 — MLP for Time Series

### What is an MLP?

A **Multilayer Perceptron (MLP)** is a fully connected feedforward neural network. It maps
the lag feature vector $\mathbf{x} \in \mathbb{R}^p$ to a scalar forecast through stacked
layers of linear transformations and non-linear activations:

$$\mathbf{h}^{(1)} = \sigma(W^{(1)} \mathbf{x} + \mathbf{b}^{(1)})$$
$$\mathbf{h}^{(2)} = \sigma(W^{(2)} \mathbf{h}^{(1)} + \mathbf{b}^{(2)})$$
$$\hat{y} = W^{(3)} \mathbf{h}^{(2)} + b^{(3)}$$

where $\sigma(z) = \max(0, z)$ is the **ReLU** activation function.

**Key advantage:** MLPs can learn non-linear combinations of lag features.

**Key limitation:** The MLP treats the lag vector as an *unordered* set of features —
it doesn't know that `lag_1` is more recent than `lag_12`. RNNs and LSTMs (Part 6)
encode this sequential structure explicitly.

### Why do we need to scale?

Neural networks require **input scaling** because gradient updates are sensitive to feature
magnitude. We use `MinMaxScaler` fitted *only* on the training data:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \in [0, 1]$$

> ⚠️ **Scaling rule:** `fit_transform` on training data only. `transform` on test data.
> Never fit the scaler on test data — this would leak future statistics into training.

### 5.1 Scale the data

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# FILL IN: fit_transform on training data, transform on test data
X_train_s = scaler_X.fit_transform(???)   # fit AND transform train
X_test_s  = scaler_X.transform(???)       # transform test ONLY

y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()      # for reference

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32)

print('X_train_t shape:', X_train_t.shape)
print('y_train_t shape:', y_train_t.shape)

### 5.2 Define the MLP architecture

**Your turn!** Complete the `MLPForecaster` class below. The network should have:
- A linear layer mapping `input_size` → `hidden_size`, followed by ReLU
- A linear layer mapping `hidden_size` → `hidden_size // 2`, followed by ReLU
- An output linear layer mapping `hidden_size // 2` → 1

> 💡 **PyTorch pattern:** Define layers in `__init__` using `nn.Sequential`. Implement
> the forward pass in `forward(self, x)`. `nn.Linear(in, out)` creates a fully connected
> layer. `nn.ReLU()` is the activation.

In [ ]:
class MLPForecaster(nn.Module):
    """
    Simple Multilayer Perceptron for time series forecasting.

    Takes a lag feature vector of length `input_size` and produces
    a single scalar forecast.

    Parameters
    ----------
    input_size  : int, number of lag features
    hidden_size : int, number of neurons in each hidden layer
    """
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, ???),       # FILL IN
            nn.ReLU(),
            nn.Linear(???, hidden_size // 2), # FILL IN
            nn.ReLU(),
            nn.Linear(hidden_size // 2, ???)  # FILL IN: output 1 value
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

### 5.3 Train the MLP

In [ ]:
mlp     = MLPForecaster(input_size=N_LAGS, hidden_size=32)
opt     = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 300
losses = []

for epoch in range(EPOCHS):
    mlp.train()
    opt.zero_grad()
    # FILL IN: forward pass and loss computation
    pred = mlp(???)
    loss = loss_fn(???, y_train_t)
    loss.backward()
    opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(losses)
ax.set_title('MLP Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (scaled)')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the MLP
mlp.eval()
with torch.no_grad():
    mlp_preds_s = mlp(X_test_t).numpy()    # predictions in scaled space

# FILL IN: inverse-transform predictions back to original (log-diff) scale
mlp_preds = scaler_y.inverse_transform(???).ravel()

results['MLP'] = compute_metrics(y_test, mlp_preds, 'MLP')

---
## Part 6 — RNN & LSTM: Hidden State, Recurrence, and Gating

### Why sequence models?

The MLP receives a flat lag vector and treats lags as **unordered** features. A **Recurrent
Neural Network (RNN)** processes the lag sequence *one step at a time*, maintaining a
**hidden state** that is updated at each step. This allows the model to encode the
sequential structure of time explicitly.

### The RNN Recurrence Equation

At each time step $t$ within the sequence:

$$\mathbf{h}_t = \tanh(W_{hh}\,\mathbf{h}_{t-1} + W_{xh}\,\mathbf{x}_t + \mathbf{b}_h)$$

where:
- $\mathbf{x}_t$ — the input (here, the value of the series at lag $t$)
- $\mathbf{h}_t$ — the **hidden state**: a learned summary of the sequence so far
- $W_{hh}$ — **recurrent weights** connecting consecutive hidden states
- $W_{xh}$ — **input weights**
- $\mathbf{h}_0 = \mathbf{0}$ (initialized to zero)

After processing all $p$ lags, the final hidden state $\mathbf{h}_p$ is passed to an
output layer:

$$\hat{y} = W_o \mathbf{h}_p + b_o$$

### The Vanishing Gradient Problem

During training, gradients must flow backward through the recurrence. If the recurrent
weights are small, the gradient shrinks exponentially over time steps — this is the
**vanishing gradient problem**, which prevents RNNs from learning long-range dependencies.

### The LSTM Solution

The **LSTM (Long Short-Term Memory)** introduces a **cell state** $\mathbf{c}_t$ and
three **gates** that control what information is retained, discarded, and output:

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate — what to erase from cell state)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate — what new info to write)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate — what to expose as hidden state)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state update)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state output)}$$

The cell state $\mathbf{c}_t$ acts as a "conveyor belt" — it can carry information
across many steps without vanishing, because the **forget gate allows additive gradient
flow** instead of multiplicative.

### 6.1 Implement the RNN

**Your turn!** Complete the `RNNForecaster` class below.

> 💡 **PyTorch note:** `nn.RNN(input_size, hidden_size, batch_first=True)` processes a
> sequence tensor of shape `(batch, seq_len, input_size)`. The output `(output, h_n)` gives
> all hidden states and the final hidden state. We unsqueeze the lag vector to add the
> `input_size=1` dimension: `x.unsqueeze(-1)` → `(batch, n_lags, 1)`.

In [ ]:
class RNNForecaster(nn.Module):
    """
    Single-layer Elman RNN for time series forecasting.

    Processes the lag sequence step-by-step and produces a single
    scalar forecast from the final hidden state.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.RNN layer with batch_first=True
        self.rnn = nn.RNN(???, ???, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc  = nn.Linear(???, ???)

    def forward(self, x):
        # x shape: (batch, n_lags) → add feature dimension
        x = x.unsqueeze(-1)           # (batch, n_lags, 1)
        # FILL IN: pass through RNN, extract final hidden state h_n
        _, h_n = self.rnn(???)
        h_n = h_n.squeeze(0)          # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

### 6.2 Implement the LSTM

> 💡 **Note:** `nn.LSTM` returns `(output, (h_n, c_n))` — the tuple includes both the
> hidden state and the cell state. We use only `h_n` for the output prediction.

In [ ]:
class LSTMForecaster(nn.Module):
    """
    Single-layer LSTM for time series forecasting.

    Extends the RNN with a cell state and gating mechanisms, allowing
    the model to selectively retain or forget information across lags.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden/cell state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.LSTM layer with batch_first=True
        self.lstm = nn.LSTM(???, ???, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc   = nn.Linear(???, ???)

    def forward(self, x):
        x = x.unsqueeze(-1)               # (batch, n_lags, 1)
        # FILL IN: pass through LSTM, extract h_n from the returned tuple
        _, (h_n, _) = self.lstm(???)
        h_n = h_n.squeeze(0)              # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

### 6.3 Train the RNN and LSTM

The function below trains any PyTorch model with MSE loss. Complete it, then train both
the RNN and LSTM.

### 🤝 Dividir y Confluir

Your instructor will assign your group a `hidden_size` to experiment with:

| Group | Model | hidden_size |
|---|---|---|
| A | RNN | 16 |
| B | RNN | 64 |
| C | LSTM | 16 |
| D | LSTM | 64 |

Fit your assigned configuration, compute RMSE/MAE, and share results for class comparison.

In [ ]:
def train_model(model, X_t, y_t, epochs=300, lr=1e-3, label='Model'):
    """
    Train a PyTorch model with MSE loss and Adam optimizer.

    Parameters
    ----------
    model  : nn.Module
    X_t    : torch.Tensor, (n_samples, n_lags)
    y_t    : torch.Tensor, (n_samples,)
    epochs : int
    lr     : float, learning rate
    label  : str, for display

    Returns
    -------
    list of training losses
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    losses    = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        # FILL IN: forward pass and loss
        pred = model(???)
        loss = loss_fn(???, y_t)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())

    print(f'{label} — final loss: {losses[-1]:.6f}')
    return losses


rnn_model  = RNNForecaster(hidden_size=32)
lstm_model = LSTMForecaster(hidden_size=32)

rnn_losses  = train_model(rnn_model,  X_train_t, y_train_t, epochs=300, label='RNN')
lstm_losses = train_model(lstm_model, X_train_t, y_train_t, epochs=300, label='LSTM')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rnn_losses,  label='RNN',  color='steelblue')
ax.plot(lstm_losses, label='LSTM', color='crimson')
ax.set_title('Training Loss — RNN vs LSTM')
ax.set_xlabel('Epoch')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def eval_model(model, X_t, scaler_y):
    """
    Generate predictions and inverse-transform to original scale.

    Parameters
    ----------
    model    : trained nn.Module
    X_t      : torch.Tensor, test features
    scaler_y : fitted MinMaxScaler for the target

    Returns
    -------
    np.ndarray of predictions in original (log-diff) scale
    """
    model.eval()
    with torch.no_grad():
        # FILL IN: run model on test features
        preds_s = model(???).numpy()
    # FILL IN: inverse-transform
    return scaler_y.inverse_transform(preds_s.reshape(-1, 1)).ravel()


rnn_preds  = eval_model(rnn_model,  X_test_t, scaler_y)
lstm_preds = eval_model(lstm_model, X_test_t, scaler_y)

results['RNN']  = compute_metrics(y_test, rnn_preds,  'RNN')
results['LSTM'] = compute_metrics(y_test, lstm_preds, 'LSTM')

### ✏️ Written Response 6

Answer each question in 2–4 sentences:

1. What is the **hidden state** in an RNN, and what role does it play that the MLP's
   lag vector does not?
2. What is the **vanishing gradient problem** in plain language? Why does it make long-range
   dependencies hard to learn?
3. Pick one LSTM gate (forget, input, or output). Describe in your own words what it does
   and why it helps compared to a plain RNN.
4. On this dataset, did the LSTM outperform the RNN by a large margin? Why or why not?

> **YOUR ANSWER:**

---
## Part 7 — Full Model Comparison

### 7.1 Build the comparison table

**Your turn!** Assemble all results into a sorted DataFrame.

In [ ]:
# FILL IN: create a DataFrame from the results dict and sort by RMSE
comparison_df = pd.DataFrame(???).T
comparison_df = comparison_df.sort_values(???)

print('\n=== Model Comparison (sorted by RMSE) ===')
print(comparison_df.to_string(float_format='{:.5f}'.format))

In [ ]:
# Visualise all predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
test_range = np.arange(len(y_test))

# Top panel: classical + tree models
axes[0].plot(test_range, y_test,          label='Actual',            color='black',      linewidth=2)
axes[0].plot(test_range, naive_preds,     label='Naive',             color='gray',       linestyle='--')
axes[0].plot(test_range, ar_preds.values, label='AR(12)',            color='steelblue',  linestyle='-.')
axes[0].plot(test_range, rf_preds,        label='Random Forest',     color='forestgreen')
axes[0].plot(test_range, gb_preds,        label='Gradient Boosting', color='darkorange')
axes[0].set_title('Classical & Tree-Based Models')
axes[0].legend(fontsize=9)

# Bottom panel: neural models
axes[1].plot(test_range, y_test,          label='Actual', color='black',  linewidth=2)
axes[1].plot(test_range, mlp_preds,       label='MLP',    color='purple')
axes[1].plot(test_range, rnn_preds,       label='RNN',    color='teal')
axes[1].plot(test_range, lstm_preds,      label='LSTM',   color='crimson')
axes[1].set_title('Neural Models')
axes[1].set_xlabel('Test Step')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 4))
models    = list(comparison_df.index)
rmse_vals = comparison_df['RMSE'].values
mae_vals  = comparison_df['MAE'].values

x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, rmse_vals, w, label='RMSE', color='steelblue', alpha=0.8)
ax.bar(x + w/2, mae_vals,  w, label='MAE',  color='crimson',   alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_title('Model Comparison: RMSE and MAE on Test Set')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Final Written Reflection

Write a **4–6 sentence summary** as if reporting to a colleague who did not attend today.
Your summary must address:

- Why temporal ordering is critical for train/test splits in time series
- What the naive baseline tells you and why it matters
- The key algorithmic difference between the MLP and the RNN/LSTM
- How gating in the LSTM addresses the vanishing gradient problem
- Which model performed best on this dataset and one hypothesis for why

**Share your summary with the class (dividir y confluir debrief).**

> **YOUR ANSWER:**

---

## Moving Forward

Choose a different publicly available time series (e.g., daily stock prices, electricity
consumption, or temperature data) and run the complete pipeline from today:

1. Apply appropriate transformations to achieve stationarity
2. Build a lag-embedded feature matrix with a chosen `n_lags`
3. Create a time-aware train/test split
4. Train at least three of the six model families from today
5. Report RMSE and MAE for each model and interpret the results
6. Write 3–5 sentences comparing your findings to the Air Passengers results

```python
# Your code here
```

---

## References / Further Reading

* [Forecasting: Principles and Practice — Chapter 5 (Regression models)](https://otexts.com/fpp3/regression.html)
* [Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow — Chapter 15 (RNNs)](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/)
* [Understanding LSTM Networks — Colah's Blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)
* [PyTorch `nn.RNN` documentation](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
* [PyTorch `nn.LSTM` documentation](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* [Scikit-Learn `RandomForestRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)